# Laboratorio · Tema 6 — Fine-tuning de un Transformer
### Aprendizaje Profundo · CUNEF · Prof. Óscar Sánchez Rueda

En este laboratorio vas a **ajustar (fine-tuning)** un Transformer **preentrenado** 
(*DistilBERT*) para una tarea de **clasificación de texto**: decidir si una crítica de 
cine es **positiva** o **negativa**.

Usaremos **Keras / TensorFlow** con la librería 🤗 **Transformers** de Hugging Face.

> **Recomendado:** ejecútalo en **Google Colab** con GPU 
(*Entorno de ejecución → Cambiar tipo de entorno → GPU*).

**Objetivos**
1. Cargar un dataset de texto etiquetado.
2. Tokenizar con el tokenizer del modelo preentrenado.
3. Cargar `DistilBERT` y ajustarlo con `model.fit` (Keras).
4. Evaluar y probar el modelo con frases nuevas.


## 0. Instalación
Instalamos las librerías (en Colab, esto tarda ~1 min).


In [ ]:
!pip install -q transformers datasets


## 1. Cargar los datos
Usamos **IMDB** (críticas de cine en inglés, etiquetadas positivo/negativo). 
Para que el laboratorio sea rápido en clase, tomamos un **subconjunto**.


In [ ]:
from datasets import load_dataset

imdb = load_dataset('imdb')

# Subconjunto para ir rápido (sube estos números si tienes GPU y tiempo)
train_ds = imdb['train'].shuffle(seed=42).select(range(2000))
test_ds  = imdb['test'].shuffle(seed=42).select(range(1000))

print(train_ds)
print('Ejemplo:', train_ds[0]['label'], '-', train_ds[0]['text'][:200])


## 2. Tokenizar
Cada modelo trae su **tokenizer**: convierte el texto en los identificadores que la red entiende. 
Usamos el de `distilbert-base-uncased`.


In [ ]:
from transformers import AutoTokenizer

checkpoint = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_tok = train_ds.map(tokenize, batched=True)
test_tok  = test_ds.map(tokenize, batched=True)


## 3. Preparar los `tf.data.Dataset`
Convertimos los datos al formato que espera Keras.


In [ ]:
import tensorflow as tf

cols = ['input_ids', 'attention_mask']
train_tok.set_format('tensorflow', columns=cols + ['label'])
test_tok.set_format('tensorflow', columns=cols + ['label'])

def to_tf(ds, shuffle):
    x = {k: ds[k] for k in cols}
    d = tf.data.Dataset.from_tensor_slices((x, ds['label']))
    if shuffle: d = d.shuffle(1000)
    return d.batch(16)

train_tf = to_tf(train_tok, True)
test_tf  = to_tf(test_tok, False)


## 4. Cargar el modelo preentrenado
`TFAutoModelForSequenceClassification` carga DistilBERT y le añade una **cabeza de clasificación** 
(2 clases). Solo esa cabeza empieza de cero; el resto ya sabe 'entender' inglés.


In [ ]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)


## 5. Compilar y entrenar (fine-tuning)
Un **learning rate pequeño** (2e-5) es clave: no queremos destrozar lo ya aprendido, solo ajustarlo. 
Con 1 época suele bastar para ver el efecto.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

model.compile(
    optimizer=Adam(learning_rate=2e-5),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

history = model.fit(train_tf, validation_data=test_tf, epochs=1)


## 6. Probar con frases nuevas
Pasamos texto por el tokenizer y por el modelo, y aplicamos softmax a los *logits*.


In [ ]:
import numpy as np

frases = [
    'An absolute masterpiece, I loved every minute of it.',
    'Terrible movie, a complete waste of time.',
]
enc = tokenizer(frases, truncation=True, padding=True, max_length=256, return_tensors='tf')
logits = model(enc).logits
probs = tf.nn.softmax(logits, axis=-1).numpy()

for f, p in zip(frases, probs):
    etiqueta = 'POSITIVA' if p[1] > p[0] else 'NEGATIVA'
    print(f'{etiqueta} ({p.max():.2%}) -> {f}')


## 7. Tu turno
- Sube el tamaño del subconjunto (o usa el dataset completo) y compara la *accuracy*.
- Prueba con **tus propias frases** en la celda anterior.
- Cambia el `checkpoint` a otro modelo (p. ej. `bert-base-uncased`) y compara.
- **Reflexiona:** ¿por qué basta con tan pocos datos y una época? (Pista: *transfer learning*.)
